In [ ]:
import numpy as np
import rasterio
from scipy.ndimage import sobel


def calculate_slope(dem: np.ndarray, resolution: float) -> np.ndarray:
    """
    Computes the slope from the Digital Elevation Model (DEM).

    Parameters:
        dem (numpy.ndarray): 2D array representing the DEM.
        resolution (float): Spatial resolution of the DEM (pixel size in meters).

    Returns:
        numpy.ndarray: 2D array containing slope values in degrees.
    """
    dx = sobel(dem, axis=1) / (8.0 * resolution)  # East-West gradient (X direction)
    dy = sobel(dem, axis=0) / (8.0 * resolution)  # North-South gradient (Y direction)
    
    slope_rad = np.arctan(np.sqrt(dx**2 + dy**2))
    slope_deg = np.degrees(slope_rad)
    
    return slope_deg


def calculate_profile_convexity(dem: np.ndarray, resolution: float) -> np.ndarray:
    """
    Computes the profile convexity from the Digital Elevation Model (DEM).

    Parameters:
        dem (numpy.ndarray): 2D array representing the DEM.
        resolution (float): Spatial resolution of the DEM (pixel size in meters).

    Returns:
        numpy.ndarray: 2D array containing profile convexity values.
    """
    dzdx = sobel(dem, axis=1) / (8.0 * resolution)  # First derivative in X direction
    dzdy = sobel(dem, axis=0) / (8.0 * resolution)  # First derivative in Y direction
    
    aspect = np.arctan2(dzdy, dzdx)  # Slope aspect (orientation relative to North)
    
    d2zdx2 = sobel(dzdx, axis=1) / (8.0 * resolution)  # Second derivative in X direction
    d2zdy2 = sobel(dzdy, axis=0) / (8.0 * resolution)  # Second derivative in Y direction
    
    profile_convexity = d2zdx2 * np.cos(aspect) ** 2 + d2zdy2 * np.sin(aspect) ** 2
    
    return profile_convexity


# File paths
dem_path = "/home/stagiaire/Téléchargements/rasters_COP30/cop_dem_fin.tif"
slope_output_path = "slope_output.tif"
convexity_output_path = "profile_convexity_output.tif"

# Process DEM
with rasterio.open(dem_path) as dem_dataset:
    dem = dem_dataset.read(1)  # Read the first band (assumed to be the DEM)
    resolution = dem_dataset.transform[0]  # Extract pixel resolution
    
    slope = calculate_slope(dem, resolution)
    profile_convexity = calculate_profile_convexity(dem, resolution)
    
    profile = dem_dataset.profile
    profile.update(dtype=rasterio.float32, count=1, nodata=None)
    
    # Save slope raster
    with rasterio.open(slope_output_path, 'w', **profile) as slope_dataset:
        slope_dataset.write(slope.astype(rasterio.float32), 1)
    
    # Save profile convexity raster
    with rasterio.open(convexity_output_path, 'w', **profile) as convexity_dataset:
        convexity_dataset.write(profile_convexity.astype(rasterio.float32), 1)

print(f"Slope file saved as: {slope_output_path}")
print(f"Profile convexity file saved as: {convexity_output_path}")
